In [1]:
import pandas as pd
from pathlib import Path

# Project root
project_root = Path.cwd().parent

# Load the processed subset
subset_path = project_root / "data" / "processed" / "catalog_subset.parquet"

catalog_df = pd.read_parquet(subset_path)

print("Dataset shape:", catalog_df.shape)
catalog_df.head()

Dataset shape: (7900, 6)


,item_id,title,product_type,brand,color,target_category
0,B008U4QK1G,strathwood rhodes 深坐 Loveseat,SOFA,Strathwood by Amazon.com,Black,Furniture
1,B07FQP8YQ4,"Amazon Essentials Sandalia de tira para mujer,...",SANDAL,Amazon Essentials,Color piel (nude),Footwear
2,B086XG41S4,"AmazonCommercial Gerillte Saftgläser, Barzubeh...",DRINKING_CUP,AmazonCommercial,Durchsichtig,Kitchenware
3,B071JM376C,Red Wagon Boys’ Velcro Low-Top Trainers,SHOES,Red Wagon,None,Footwear
4,B07P87VSD6,"Rivet - Taburete de Bar con Remaches, 96,5 cm ...",HOME_FURNITURE_AND_DECOR,Rivet,None,Furniture


In [3]:
import torch
import open_clip

# Device (CPU only)
device = "cpu"

# Load model
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)

tokenizer = open_clip.get_tokenizer("ViT-B-32")

model.eval()
model.to(device)

print("OpenCLIP model loaded successfully!")
print("Device:", device)

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

d:\christ college\sem 4\project\multimodel project\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--timm--vit_base_patch32_clip_224.openai. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\christ college\sem 4\project\multimodel project\.venv\Lib\site-packages\op

OpenCLIP model loaded successfully!
Device: cpu


In [4]:
import numpy as np
from tqdm import tqdm

titles = catalog_df["title"].fillna("").tolist()

print(f"Generating embeddings for {len(titles)} titles...")

batch_size = 64
text_embeddings = []

with torch.no_grad():
    for i in tqdm(range(0, len(titles), batch_size)):
        batch_titles = titles[i:i + batch_size]
        
        # Tokenize
        text_tokens = tokenizer(batch_titles).to(device)
        
        # Encode
        text_features = model.encode_text(text_tokens)
        
        # Normalize
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        
        text_embeddings.append(text_features.cpu().numpy())

# Combine all batches
text_embeddings = np.vstack(text_embeddings)

print()
print("Text embeddings shape:", text_embeddings.shape)
print("Embedding dtype:", text_embeddings.dtype)

Generating embeddings for 7900 titles...


100%|██████████| 124/124 [22:07<00:00, 10.70s/it]


Text embeddings shape: (7900, 512)
Embedding dtype: float32


In [5]:
import os

# Create embeddings directory
emb_dir = project_root / "embeddings"
emb_dir.mkdir(exist_ok=True)

# Save embeddings
text_emb_path = emb_dir / "text_embeddings.npy"

np.save(text_emb_path, text_embeddings)

# Save metadata mapping
metadata_path = emb_dir / "embedding_metadata.parquet"

catalog_df[["item_id", "title", "target_category"]].to_parquet(
    metadata_path,
    index=False
)

print(f"Saved embeddings to: {text_emb_path}")
print(f"Saved metadata to: {metadata_path}")

size_mb = os.path.getsize(text_emb_path) / (1024 * 1024)
print(f"Embedding file size: {size_mb:.2f} MB")

Saved embeddings to: d:\christ college\sem 4\project\multimodel project\embeddings\text_embeddings.npy
Saved metadata to: d:\christ college\sem 4\project\multimodel project\embeddings\embedding_metadata.parquet
Embedding file size: 15.43 MB


In [6]:
# Load back and verify
loaded_embeddings = np.load(text_emb_path)

print("Loaded shape:", loaded_embeddings.shape)

# Check normalization
norms = np.linalg.norm(loaded_embeddings, axis=1)

print("Min norm:", norms.min())
print("Max norm:", norms.max())
print("Mean norm:", norms.mean())

Loaded shape: (7900, 512)
Min norm: 0.9999997
Max norm: 1.0000002
Mean norm: 1.0


Create the FAISS index

In [7]:
import faiss
import numpy as np

# Embedding dimension
embedding_dim = text_embeddings.shape[1]

print("Embedding dimension:", embedding_dim)

# Create cosine-similarity index
# Since embeddings are normalized, Inner Product = Cosine Similarity
index = faiss.IndexFlatIP(embedding_dim)

# Add embeddings
index.add(text_embeddings)

print("FAISS index created successfully!")
print("Total vectors indexed:", index.ntotal)

Embedding dimension: 512
FAISS index created successfully!
Total vectors indexed: 7900


In [8]:
# Save FAISS index
faiss_path = project_root / "embeddings" / "faiss.index"

faiss.write_index(index, str(faiss_path))

print(f"FAISS index saved to: {faiss_path}")

FAISS index saved to: d:\christ college\sem 4\project\multimodel project\embeddings\faiss.index


In [9]:
def search_similar_products(query_text, top_k=5):
    """
    Search for semantically similar products using CLIP text embeddings.
    """
    
    # Encode query
    with torch.no_grad():
        tokens = tokenizer([query_text]).to(device)
        query_emb = model.encode_text(tokens)
        query_emb = query_emb / query_emb.norm(dim=-1, keepdim=True)
        query_emb = query_emb.cpu().numpy().astype(np.float32)
    
    # Search
    scores, indices = index.search(query_emb, top_k)
    
    # Build results
    results = []
    
    for score, idx in zip(scores[0], indices[0]):
        row = catalog_df.iloc[idx]
        
        results.append({
            "score": float(score),
            "title": row["title"],
            "category": row["target_category"],
            "product_type": row["product_type"]
        })
    
    return pd.DataFrame(results)

In [10]:
# Test query 1
search_similar_products(
    "women running shoes",
    top_k=5
)

,score,title,category,product_type
0,0.888447,find. V1241a Women's Trainers,Footwear,SHOES
1,0.888083,Amazon Brand - find. Women's Trainers,Footwear,SHOES
2,0.885461,Klepe Men's Running Shoes,Footwear,SHOES
3,0.885461,Klepe Men's Running Shoes,Footwear,SHOES
4,0.875791,Amazon Brand - Symbol Women's Sneakers,Footwear,SHOES


In [11]:
# Test query 2
search_similar_products(
    "wooden dining chair",
    top_k=5
)

,score,title,category,product_type
0,0.885213,Movian Wendy Dining Chair,Furniture,CHAIR
1,0.885213,Movian Wendy Dining Chair,Furniture,CHAIR
2,0.858033,"Stone & Beam Dining Chair 36.5""H,-",Furniture,CHAIR
3,0.854152,Rivet Samson Dining Chair,Furniture,CHAIR
4,0.854152,Rivet Samson Dining Chair,Furniture,CHAIR


In [12]:
# Test query 3
search_similar_products(
    "glass juice cup",
    top_k=5
)

,score,title,category,product_type
0,0.817433,Set of 6 Highball Clear Glass 350ml Water Juic...,Kitchenware,KITCHEN
1,0.805520,Pump,Kitchenware,KITCHEN
2,0.803135,"365 Everyday Value, Plastic Cups (16 fl oz), 2...",Kitchenware,DRINKING_CUP
3,0.803135,"365 Everyday Value, Plastic Cups (16 fl oz), 2...",Kitchenware,DRINKING_CUP
4,0.800605,"AmazonCommercial glas mätkopp, 1 cup, genomski...",Kitchenware,MEASURING_CUP


In [13]:
def find_duplicates(product_index, top_k=6):
    """
    Find near-duplicate products for a given catalog index.
    top_k=6 because the first result is the product itself.
    """
    
    # Get embedding
    query_emb = text_embeddings[product_index:product_index+1]
    
    # Search
    scores, indices = index.search(query_emb, top_k)
    
    # Build results (skip the first result = self)
    results = []
    
    for score, idx in zip(scores[0][1:], indices[0][1:]):
        row = catalog_df.iloc[idx]
        
        results.append({
            "score": float(score),
            "item_id": row["item_id"],
            "title": row["title"],
            "category": row["target_category"]
        })
    
    return pd.DataFrame(results)

In [14]:
# Find a shoe product
shoe_idx = catalog_df[
    catalog_df["title"].str.contains("Running Shoes|Trainers|Sneakers", case=False, na=False)
].index[0]

print("Query product:")
print(catalog_df.loc[shoe_idx, ["title", "target_category"]])

print()
print("Potential duplicates:")

find_duplicates(shoe_idx)

Query product:
title              Red Wagon Boys’ Velcro Low-Top Trainers
target_category                                   Footwear
Name: 3, dtype: object

Potential duplicates:


,score,item_id,title,category
0,0.880934,B071JM37MZ,RED WAGON Boys’ Velcro Low-Top Trainers Black ...,Footwear
1,0.874100,B01MSEMSOP,Amazon-merk: RED WAGON jongens High-Top sneakers,Footwear
2,0.861755,B072LSGXC2,RED WAGON meisjes platte sneaker,Footwear
3,0.858642,B0732WSYTS,Red Wagon Kids' Tom Boots,Footwear
4,0.858491,B0732W4X7D,Amazon Brand - RED WAGON Boy's Desert Boots,Footwear


In [15]:
DUPLICATE_THRESHOLD = 0.92

duplicates_df = find_duplicates(shoe_idx)

high_conf_duplicates = duplicates_df[
    duplicates_df["score"] >= DUPLICATE_THRESHOLD
]

print(f"Duplicate threshold: {DUPLICATE_THRESHOLD}")
print()
print("High-confidence duplicates:")

high_conf_duplicates

Duplicate threshold: 0.92

High-confidence duplicates:


,score,item_id,title,category


In [16]:
# Check score distribution
duplicates_df

,score,item_id,title,category
0,0.880934,B071JM37MZ,RED WAGON Boys’ Velcro Low-Top Trainers Black ...,Footwear
1,0.874100,B01MSEMSOP,Amazon-merk: RED WAGON jongens High-Top sneakers,Footwear
2,0.861755,B072LSGXC2,RED WAGON meisjes platte sneaker,Footwear
3,0.858642,B0732WSYTS,Red Wagon Kids' Tom Boots,Footwear
4,0.858491,B0732W4X7D,Amazon Brand - RED WAGON Boy's Desert Boots,Footwear


In [17]:
def duplicate_search_api(query_text, threshold=0.90, top_k=5):
    """
    API-style duplicate search from raw text.
    """
    
    results = search_similar_products(query_text, top_k=top_k)
    
    results["is_duplicate"] = results["score"] >= threshold
    
    return results

In [18]:
duplicate_search_api(
    "women trainers sneakers",
    threshold=0.88,
    top_k=5
)

,score,title,category,product_type,is_duplicate
0,0.931372,Amazon Brand - find. Women's Trainers,Footwear,SHOES,True
1,0.919083,find. V1241a Women's Trainers,Footwear,SHOES,True
2,0.917958,Amazon Brand - Symbol Women's Sneakers,Footwear,SHOES,True
3,0.917958,Amazon Brand - Symbol Women's Sneakers,Footwear,SHOES,True
4,0.882197,"Amazon Brand - Jenny Slip_on, Women’s Low-Top ...",Footwear,SHOES,True


In [19]:
# Production thresholds
DUPLICATE_THRESHOLD = 0.90
REVIEW_THRESHOLD = 0.70
MISMATCH_THRESHOLD = 0.18

print("Thresholds configured:")
print(f"Duplicate: {DUPLICATE_THRESHOLD}")
print(f"Review queue: {REVIEW_THRESHOLD}")
print(f"Mismatch: {MISMATCH_THRESHOLD}")

Thresholds configured:
Duplicate: 0.9
Review queue: 0.7
Mismatch: 0.18


In [20]:
def compute_consistency_score(image_description, product_title):
    """
    Simulate image-text consistency using CLIP text embeddings.
    Later, image_description will be replaced by a real image embedding.
    """
    
    with torch.no_grad():
        tokens = tokenizer([image_description, product_title]).to(device)
        features = model.encode_text(tokens)
        features = features / features.norm(dim=-1, keepdim=True)
        features = features.cpu().numpy()
    
    # Cosine similarity
    score = float(np.dot(features[0], features[1]))
    
    return score

In [21]:
correct_score = compute_consistency_score(
    image_description="women running sneakers with rubber sole",
    product_title="Amazon Brand - find. Women's Trainers"
)

print(f"Correct product consistency score: {correct_score:.4f}")

Correct product consistency score: 0.8712


In [22]:
mismatch_score = compute_consistency_score(
    image_description="wireless computer mouse black color",
    product_title="Amazon Brand - find. Women's Trainers"
)

print(f"Mismatched product consistency score: {mismatch_score:.4f}")

Mismatched product consistency score: 0.7146


In [23]:
def check_mismatch(image_description, product_title, threshold=0.18):
    score = compute_consistency_score(image_description, product_title)
    
    return {
        "score": round(score, 4),
        "is_mismatch": score < threshold,
        "threshold": threshold
    }

In [24]:
print("Correct case:")
print(check_mismatch(
    "women running sneakers with rubber sole",
    "Amazon Brand - find. Women's Trainers"
))

print()

print("Mismatch case:")
print(check_mismatch(
    "wireless computer mouse black color",
    "Amazon Brand - find. Women's Trainers"
))

Correct case:
{'score': 0.8712, 'is_mismatch': False, 'threshold': 0.18}

Mismatch case:
{'score': 0.7146, 'is_mismatch': False, 'threshold': 0.18}


In [25]:
# Updated threshold for text-vs-text simulation
MISMATCH_THRESHOLD = 0.80

print("Updated mismatch threshold:", MISMATCH_THRESHOLD)

Updated mismatch threshold: 0.8


In [26]:
def check_mismatch(image_description, product_title, threshold=MISMATCH_THRESHOLD):
    score = compute_consistency_score(image_description, product_title)
    
    return {
        "score": round(score, 4),
        "is_mismatch": score < threshold,
        "threshold": threshold
    }

In [27]:
print("Correct case:")
print(check_mismatch(
    "women running sneakers with rubber sole",
    "Amazon Brand - find. Women's Trainers"
))

print()

print("Mismatch case:")
print(check_mismatch(
    "wireless computer mouse black color",
    "Amazon Brand - find. Women's Trainers"
))

Correct case:
{'score': 0.8712, 'is_mismatch': False, 'threshold': 0.8}

Mismatch case:
{'score': 0.7146, 'is_mismatch': True, 'threshold': 0.8}


In [28]:
import sqlite3
from datetime import datetime

# Database path
db_path = project_root / "services" / "review_queue.db"
db_path.parent.mkdir(exist_ok=True)

# Connect
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS review_queue (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    item_id TEXT,
    title TEXT,
    category TEXT,
    confidence REAL,
    mismatch_score REAL,
    duplicate_score REAL,
    reason TEXT,
    created_at TEXT
)
""")

conn.commit()

print(f"Database created at: {db_path}")

Database created at: d:\christ college\sem 4\project\multimodel project\services\review_queue.db


In [29]:
def add_to_review_queue(
    item_id,
    title,
    category,
    confidence,
    mismatch_score,
    duplicate_score,
    reason
):
    cursor.execute(
        """
        INSERT INTO review_queue
        (item_id, title, category, confidence,
         mismatch_score, duplicate_score, reason, created_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            item_id,
            title,
            category,
            confidence,
            mismatch_score,
            duplicate_score,
            reason,
            datetime.now().isoformat()
        )
    )
    
    conn.commit()

# Add a sample flagged item
add_to_review_queue(
    item_id="TEST_001",
    title="Amazon Brand - find. Women's Trainers",
    category="Footwear",
    confidence=0.62,
    mismatch_score=0.71,
    duplicate_score=0.93,
    reason="Low confidence + possible duplicate"
)

print("Sample item added to review queue")

Sample item added to review queue


In [30]:
review_df = pd.read_sql_query(
    "SELECT * FROM review_queue ORDER BY created_at DESC",
    conn
)

review_df

,id,item_id,title,category,confidence,mismatch_score,duplicate_score,reason,created_at
0,1,TEST_001,Amazon Brand - find. Women's Trainers,Footwear,0.62,0.71,0.93,Low confidence + possible duplicate,2026-07-23T18:51:35.696223


In [31]:
def should_send_to_review(confidence, mismatch_score, duplicate_score):
    reasons = []
    
    if confidence < 0.70:
        reasons.append("Low confidence")
    
    if mismatch_score < 0.80:
        reasons.append("Possible image-text mismatch")
    
    if duplicate_score >= 0.90:
        reasons.append("Possible duplicate product")
    
    return {
        "needs_review": len(reasons) > 0,
        "reasons": reasons
    }

In [32]:
review_check = should_send_to_review(
    confidence=0.62,
    mismatch_score=0.71,
    duplicate_score=0.93
)

print(review_check)

{'needs_review': True, 'reasons': ['Low confidence', 'Possible image-text mismatch', 'Possible duplicate product']}
